In [1]:
import json
import os
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from urllib.parse import urlparse
import msal
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(override=True)  # reload .env on every run

True

In [2]:
load_dotenv(override=True)  # pick up latest .env values

# Env var names on the left; your app IDs as defaults on the right.
TENANT_ID = os.getenv("AZURE_TENANT_ID", "ab8255df-0b3c-46e9-8c0d-11bc4b132da1")
CLIENT_ID = os.getenv("AZURE_CLIENT_ID", "764908e8-ed83-496d-8a08-e06fc28b8505")
CLIENT_SECRET = os.getenv("AZURE_CLIENT_SECRET", "")

# Root communication site: RAG AI Agentic Deployment Site (Test)
SHAREPOINT_SITE_URL = os.getenv(
    "SHAREPOINT_SITE_URL",
    "https://amenhanna.sharepoint.com",
)

LIST_NAMES: list[str] = []
LIBRARY_NAMES: list[str] = []

# "auto" = try app-only first, fall back to device_code if permissions missing
# Options: auto | client_credentials | device_code | auth_code
AUTH_MODE = os.getenv("GRAPH_AUTH_MODE", "auto")

# Prompt once if secret is still missing and client_credentials is required
if AUTH_MODE == "client_credentials" and not CLIENT_SECRET:
    CLIENT_SECRET = getpass("Azure client secret (paste from Portal → Certificates & secrets): ")

REDIRECT_URI = os.getenv("AZURE_REDIRECT_URI", "http://localhost:8400")

GRAPH_BASE = "https://graph.microsoft.com/v1.0"
SCOPES = ["https://graph.microsoft.com/.default"]
DELEGATED_SCOPES = ["Sites.Read.All", "Files.Read.All"]

PROJECT_DIR = Path.cwd()
JSON_OUTPUT_DIR = PROJECT_DIR / "data" / "sharepoint_json"
JSON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not TENANT_ID or not CLIENT_ID:
    raise ValueError(
        "TENANT_ID and CLIENT_ID must be set. "
        "Use AZURE_TENANT_ID / AZURE_CLIENT_ID env vars or edit the defaults above."
    )

if AUTH_MODE == "client_credentials" and not CLIENT_SECRET:
    raise ValueError(
        "No client secret provided. Either:\n"
        "  1. Create a .env file with AZURE_CLIENT_SECRET=your-secret\n"
        "  2. Set AUTH_MODE=auto or AUTH_MODE=device_code in .env"
    )

print(f"Tenant: {TENANT_ID}")
print(f"Client: {CLIENT_ID}")
print(f"Auth mode: {AUTH_MODE}")
print(f"Client secret: {'set' if CLIENT_SECRET else 'missing'}")
print(f"SharePoint site: {SHAREPOINT_SITE_URL}")
print(f"JSON output: {JSON_OUTPUT_DIR}")

Tenant: ab8255df-0b3c-46e9-8c0d-11bc4b132da1
Client: 764908e8-ed83-496d-8a08-e06fc28b8505
Auth mode: auto
Client secret: set
SharePoint site: https://amenhanna.sharepoint.com
JSON output: c:\Users\natna\Documents\RAG_Project\RAG_Application-\data\sharepoint_json


## 2. Authenticate with Microsoft Graph


In [3]:
REQUIRED_GRAPH_PERMS = {"Sites.Read.All", "Files.Read.All"}


def decode_token_claims(token: str) -> dict:
    import base64

    payload = token.split(".")[1]
    payload += "=" * (-len(payload) % 4)
    return json.loads(base64.urlsafe_b64decode(payload))


def decode_token_roles(token: str) -> list[str]:
    return decode_token_claims(token).get("roles", [])


def decode_token_scopes(token: str) -> set[str]:
    scp = decode_token_claims(token).get("scp", "")
    return set(scp.split()) if scp else set()


def _acquire_client_credentials_token(authority: str) -> dict:
    if not CLIENT_SECRET:
        raise ValueError("CLIENT_SECRET required for client_credentials auth.")
    app = msal.ConfidentialClientApplication(
        CLIENT_ID, authority=authority, client_credential=CLIENT_SECRET
    )
    return app.acquire_token_for_client(scopes=SCOPES)


def _acquire_device_code_token(authority: str) -> dict:
    """Device-code login — best for notebooks (no localhost redirect needed)."""
    app = msal.PublicClientApplication(CLIENT_ID, authority=authority)
    flow = app.initiate_device_flow(scopes=DELEGATED_SCOPES)
    if "user_code" not in flow:
        return {
            "error": "device_flow_failed",
            "error_description": flow.get("error_description", str(flow)),
        }

    print(flow["message"])  # e.g. visit https://microsoft.com/devicelogin and enter code
    return app.acquire_token_by_device_flow(flow)


def _acquire_auth_code_token(authority: str) -> dict:
    """Browser login with localhost redirect (needs Web redirect URI in Azure)."""
    from http.server import BaseHTTPRequestHandler, HTTPServer
    from urllib.parse import urlparse
    import webbrowser

    if not CLIENT_SECRET:
        raise ValueError("CLIENT_SECRET required for browser login.")

    app = msal.ConfidentialClientApplication(
        CLIENT_ID, authority=authority, client_credential=CLIENT_SECRET
    )

    flow = app.initiate_auth_code_flow(
        scopes=DELEGATED_SCOPES,
        redirect_uri=REDIRECT_URI,
    )

    auth_response: dict[str, str] = {}

    class CallbackHandler(BaseHTTPRequestHandler):
        def do_GET(self):
            auth_response["url"] = self.path
            self.send_response(200)
            self.send_header("Content-type", "text/html")
            self.end_headers()
            self.wfile.write(
                b"<html><body><h2>Login successful.</h2>"
                b"<p>You can close this tab and return to the notebook.</p></body></html>"
            )

        def log_message(self, format, *args):
            pass

    port = urlparse(REDIRECT_URI).port or 8400
    server = HTTPServer(("localhost", port), CallbackHandler)
    server.timeout = 300  # allow time for MFA / consent

    print("Opening browser for Microsoft sign-in...")
    print(f"If it does not open, go to: {flow['auth_uri']}")
    print(f"Waiting up to 300s for redirect to {REDIRECT_URI} ...")
    webbrowser.open(flow["auth_uri"])

    server.handle_request()
    server.server_close()

    if "url" not in auth_response:
        return {
            "error": "login_timeout",
            "error_description": (
                "Browser login timed out — localhost never received the redirect. "
                "Usually the Azure app is missing the Web redirect URI, or sign-in failed in the browser."
            ),
        }

    return app.acquire_token_by_auth_code_flow(flow, auth_response["url"])


def get_access_token() -> tuple[str, str]:
    if not TENANT_ID or not CLIENT_ID:
        raise ValueError("TENANT_ID and CLIENT_ID cannot be empty.")

    authority = f"https://login.microsoftonline.com/{TENANT_ID}"
    mode = AUTH_MODE

    if mode == "auto":
        if CLIENT_SECRET:
            result = _acquire_client_credentials_token(authority)
            if "access_token" in result:
                roles = set(decode_token_roles(result["access_token"]))
                if REQUIRED_GRAPH_PERMS.issubset(roles):
                    print(f"Auth: client_credentials | roles: {', '.join(sorted(roles))}")
                    return result["access_token"], "client_credentials"
                missing = REQUIRED_GRAPH_PERMS - roles
                print(
                    "App-only token missing permissions: "
                    + ", ".join(sorted(missing))
                    + "\nFalling back to device-code login..."
                )
            else:
                print("App-only auth failed. Falling back to device-code login...")
        mode = "device_code"

    if mode == "client_credentials":
        result = _acquire_client_credentials_token(authority)
    elif mode == "auth_code":
        result = _acquire_auth_code_token(authority)
    else:
        result = _acquire_device_code_token(authority)
        mode = "device_code"

    if "access_token" not in result:
        error = result.get("error_description", str(result))
        if mode == "auth_code":
            hint = (
                "Azure Portal setup for browser login:\n"
                f"  1. Authentication → Web → Redirect URI: {REDIRECT_URI}\n"
                "  2. API permissions → Delegated → Sites.Read.All, Files.Read.All\n"
                "Or set GRAPH_AUTH_MODE=device_code and enable public client flows."
            )
        else:
            hint = (
                "Azure Portal setup for device-code login:\n"
                "  1. Authentication → Advanced → Allow public client flows = Yes\n"
                "  2. API permissions → Delegated → Sites.Read.All, Files.Read.All (+ admin consent)"
            )
        raise RuntimeError(f"{error}\n\n{hint}")

    token = result["access_token"]

    if mode == "client_credentials":
        roles = set(decode_token_roles(token))
        missing = REQUIRED_GRAPH_PERMS - roles
        if missing:
            raise PermissionError(
                "Missing application permissions: "
                + ", ".join(sorted(missing))
                + ".\nGrant admin consent in Azure Portal, or set GRAPH_AUTH_MODE=auto in .env"
            )
        print(f"Auth: client_credentials | roles: {', '.join(sorted(roles))}")
    else:
        scopes = decode_token_scopes(token)
        print(f"Auth: {mode} | scopes: {', '.join(sorted(scopes)) or 'none'}")

    return token, mode


ACCESS_TOKEN, ACTUAL_AUTH_MODE = get_access_token()
HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
print("Authenticated with Microsoft Graph.")

App-only token missing permissions: Files.Read.All, Sites.Read.All
Falling back to device-code login...
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code EVFJBCL3W to authenticate.
Auth: device_code | scopes: Files.Read.All, Sites.Read.All, email, openid, profile
Authenticated with Microsoft Graph.


## 3. Graph helpers

In [4]:
def graph_get(url: str, params: dict | None = None) -> dict:
    """GET from Graph, following @odata.nextLink pagination."""
    items: list = []
    while url:
        response = requests.get(url, headers=HEADERS, params=params, timeout=60)
        if response.status_code == 401:
            roles = decode_token_roles(ACCESS_TOKEN)
            raise PermissionError(
                f"Graph 401 Unauthorized for {url}\n"
                f"Token roles: {roles or 'none'}\n"
                "Grant Application permissions Sites.Read.All + Files.Read.All "
                "and click 'Grant admin consent' in Azure Portal."
            )
        response.raise_for_status()
        payload = response.json()
        items.extend(payload.get("value", []))
        url = payload.get("@odata.nextLink")
        params = None
    return {"value": items}


def parse_sharepoint_site_url(site_url: str) -> tuple[str, str]:
    """Return (hostname, server-relative path) for Graph site lookup."""
    parsed = urlparse(site_url)
    hostname = parsed.netloc
    path = parsed.path.rstrip("/")
    return hostname, path


def build_site_graph_ref(hostname: str, site_path: str) -> str:
    """Build Graph site reference. Root site must be hostname:/ not hostname:"""
    if not site_path:
        return f"{hostname}:/"
    return f"{hostname}:{site_path}"


def get_site_id(site_url: str) -> str:
    hostname, site_path = parse_sharepoint_site_url(site_url)
    site_ref = build_site_graph_ref(hostname, site_path)
    url = f"{GRAPH_BASE}/sites/{site_ref}"
    response = requests.get(url, headers=HEADERS, timeout=60)
    if response.status_code == 401:
        roles = decode_token_roles(ACCESS_TOKEN)
        raise PermissionError(
            f"Graph 401 Unauthorized for {url}\n"
            f"Token roles: {roles or 'none'}\n"
            "Fix in Azure Portal → App registration → API permissions:\n"
            "  • Type: Application (not Delegated)\n"
            "  • Sites.Read.All\n"
            "  • Files.Read.All\n"
            "  • Then: Grant admin consent"
        )
    response.raise_for_status()
    site = response.json()
    print(f"Site: {site.get('displayName')} ({site['id']})")
    return site["id"]


SITE_ID = get_site_id(SHAREPOINT_SITE_URL)

Site: RAG AI Agentic Deployment Site (Test) (amenhanna.sharepoint.com,8a51a024-8241-4ca5-b911-a512d5115853,a75240b1-697f-469f-9645-41f525bc4e37)


## 4. Export SharePoint lists to JSON

In [5]:
def get_sharepoint_lists(site_id: str) -> list[dict]:
    url = f"{GRAPH_BASE}/sites/{site_id}/lists"
    return graph_get(url)["value"]


def get_list_items(site_id: str, list_id: str) -> list[dict]:
    url = f"{GRAPH_BASE}/sites/{site_id}/lists/{list_id}/items"
    params = {"expand": "fields", "$top": "200"}
    raw_items = graph_get(url, params=params)["value"]

    cleaned = []
    for item in raw_items:
        fields = item.get("fields", {})
        cleaned.append({
            "id": item.get("id"),
            "webUrl": item.get("webUrl"),
            "createdDateTime": item.get("createdDateTime"),
            "lastModifiedDateTime": item.get("lastModifiedDateTime"),
            "fields": fields,
        })
    return cleaned


def export_lists_to_json(site_id: str, list_names: list[str] | None = None) -> Path:
    all_lists = get_sharepoint_lists(site_id)
    if list_names:
        selected = [lst for lst in all_lists if lst["displayName"] in list_names]
    else:
        # Skip hidden/system lists
        selected = [lst for lst in all_lists if not lst.get("list", {}).get("hidden", False)]

    export = {
        "source": "sharepoint_list",
        "siteUrl": SHAREPOINT_SITE_URL,
        "exportedAt": datetime.now(timezone.utc).isoformat(),
        "lists": [],
    }

    for lst in selected:
        items = get_list_items(site_id, lst["id"])
        export["lists"].append({
            "listId": lst["id"],
            "displayName": lst["displayName"],
            "webUrl": lst.get("webUrl"),
            "itemCount": len(items),
            "items": items,
        })
        print(f"List '{lst['displayName']}': {len(items)} item(s)")

    output_path = JSON_OUTPUT_DIR / "sharepoint_lists.json"
    output_path.write_text(json.dumps(export, indent=2, default=str), encoding="utf-8")
    print(f"Saved: {output_path}")
    return output_path


lists_json_path = export_lists_to_json(SITE_ID, LIST_NAMES or None)

List 'After_Action_Reports': 637 item(s)
List 'Survey_Data': 256 item(s)
List 'Events': 0 item(s)
List 'Documents': 0 item(s)
List 'RAF_DOC': 15 item(s)
List 'Course_Curriculum': 40 item(s)
List 'Registration_Data': 256 item(s)
List 'Migration files': 2 item(s)
Saved: c:\Users\natna\Documents\RAG_Project\RAG_Application-\data\sharepoint_json\sharepoint_lists.json


In [6]:
# Preview list data as a flat pandas DataFrame
lists_data = json.loads(lists_json_path.read_text(encoding="utf-8"))

rows = []
for lst in lists_data["lists"]:
    for item in lst["items"]:
        row = {"list": lst["displayName"], **item["fields"]}
        rows.append(row)

lists_df = pd.DataFrame(rows)
lists_df.head()

,list,@odata.etag,Title,LinkTitle,field_1,field_2,field_3,field_4,field_5,field_6,...,_DisplayName,ParentVersionStringLookupId,ParentLeafNameLookupId,Registration_ID,First_Name,Last_Name,Rate,Rank,Registration_Date,Status
0,After_Action_Reports,"""95d12639-bfea-4fe3-8c7d-499a3d886d3b,1""",1,1,PLC,23-1,Leadership Fundamentals and Core Values,PO1 William Smith,Petty Officer Leadership Course (PLC) is desig...,205.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,After_Action_Reports,"""585cdc8d-c6e9-4d2a-9216-28dd375216fa,1""",2,2,PLC,23-1,Team Building and Communication,LCDR David White,Petty Officer Leadership Course (PLC) is desig...,205.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,After_Action_Reports,"""163d8e83-3a1d-4e90-8dc0-fe8aa32f32b4,1""",3,3,PLC,23-1,Conflict Resolution and Problem Solving,SCPO Michael Jackson,Petty Officer Leadership Course (PLC) is desig...,205.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,After_Action_Reports,"""5cc0fc61-8052-4b17-a312-b147b245705a,1""",4,4,PLC,23-1,Performance Management and Feedback,CAPT Joseph Davis,Petty Officer Leadership Course (PLC) is desig...,205.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,After_Action_Reports,"""865f4974-9d6a-4ae7-8cca-53cfefb2f80a,1""",5,5,PLC,23-1,Professional Development and Mentoring,SCPO Joseph Thompson,Petty Officer Leadership Course (PLC) is desig...,205.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Export document library to JSON

In [7]:
TEXT_EXTENSIONS = {".txt", ".md", ".csv", ".json", ".html", ".htm"}


def get_document_libraries(site_id: str) -> list[dict]:
    url = f"{GRAPH_BASE}/sites/{site_id}/drives"
    return graph_get(url)["value"]


def list_drive_items_recursive(drive_id: str, item_id: str = "root") -> list[dict]:
    url = f"{GRAPH_BASE}/drives/{drive_id}/items/{item_id}/children"
    children = graph_get(url)["value"]

    all_items = []
    for child in children:
        all_items.append(child)
        if "folder" in child:
            all_items.extend(list_drive_items_recursive(drive_id, child["id"]))
    return all_items


def download_file_text(drive_id: str, item_id: str) -> str | None:
    url = f"{GRAPH_BASE}/drives/{drive_id}/items/{item_id}/content"
    response = requests.get(url, headers=HEADERS, timeout=120)
    if response.status_code != 200:
        return None
    return response.text


def export_libraries_to_json(
    site_id: str,
    library_names: list[str] | None = None,
    include_file_content: bool = True,
) -> Path:
    drives = get_document_libraries(site_id)
    if library_names:
        drives = [d for d in drives if d["name"] in library_names]

    export = {
        "source": "sharepoint_document_library",
        "siteUrl": SHAREPOINT_SITE_URL,
        "exportedAt": datetime.now(timezone.utc).isoformat(),
        "libraries": [],
    }

    for drive in drives:
        items = list_drive_items_recursive(drive["id"])
        files = []

        for item in items:
            if "folder" in item:
                continue

            name = item.get("name", "")
            ext = Path(name).suffix.lower()
            file_record = {
                "id": item.get("id"),
                "name": name,
                "webUrl": item.get("webUrl"),
                "size": item.get("size"),
                "createdDateTime": item.get("createdDateTime"),
                "lastModifiedDateTime": item.get("lastModifiedDateTime"),
                "mimeType": item.get("file", {}).get("mimeType"),
                "content": None,
            }

            if include_file_content and ext in TEXT_EXTENSIONS:
                file_record["content"] = download_file_text(drive["id"], item["id"])

            files.append(file_record)

        export["libraries"].append({
            "driveId": drive["id"],
            "name": drive["name"],
            "webUrl": drive.get("webUrl"),
            "fileCount": len(files),
            "files": files,
        })
        print(f"Library '{drive['name']}': {len(files)} file(s)")

    output_path = JSON_OUTPUT_DIR / "sharepoint_libraries.json"
    output_path.write_text(json.dumps(export, indent=2, default=str), encoding="utf-8")
    print(f"Saved: {output_path}")
    return output_path


libraries_json_path = export_libraries_to_json(
    SITE_ID,
    LIBRARY_NAMES or None,
    include_file_content=True,
)

Library 'Documents': 0 file(s)
Library 'RAF_DOC': 15 file(s)
Library 'Migration files': 2 file(s)
Saved: c:\Users\natna\Documents\RAG_Project\RAG_Application-\data\sharepoint_json\sharepoint_libraries.json


In [8]:
# Preview document library metadata
libraries_data = json.loads(libraries_json_path.read_text(encoding="utf-8"))

file_rows = []
for lib in libraries_data["libraries"]:
    for f in lib["files"]:
        file_rows.append({
            "library": lib["name"],
            "name": f["name"],
            "size": f["size"],
            "hasContent": f["content"] is not None,
            "webUrl": f["webUrl"],
        })

files_df = pd.DataFrame(file_rows)
files_df.head()

,library,name,size,hasContent,webUrl
0,RAF_DOC,CTC_Comprehensive_Report.docx,49488,False,https://amenhanna.sharepoint.com/_layouts/15/D...
1,RAF_DOC,CTC_Comprehensive_Report.pdf,176426,False,https://amenhanna.sharepoint.com/RAF_DOC/CTC_C...
2,RAF_DOC,CTC_Presentation.pptx,40778,False,https://amenhanna.sharepoint.com/_layouts/15/D...
3,RAF_DOC,CWO_Comprehensive_Report.docx,47902,False,https://amenhanna.sharepoint.com/_layouts/15/D...
4,RAF_DOC,CWO_Comprehensive_Report.pdf,175424,False,https://amenhanna.sharepoint.com/RAF_DOC/CWO_C...


## 6. Merge into one JSON file for chunking

This combined file is the input for the pre-chunking RAG step.

In [10]:
def list_items_to_documents(lists_payload: dict) -> list[dict]:
    documents = []
    for lst in lists_payload.get("lists", []):
        for item in lst.get("items", []):
            documents.append({
                "sourceType": "sharepoint_list",
                "sourceName": lst["displayName"],
                "id": item.get("id"),
                "webUrl": item.get("webUrl"),
                "text": json.dumps(item.get("fields", {}), default=str),
            })
    return documents


def library_files_to_documents(libraries_payload: dict) -> list[dict]:
    documents = []
    for lib in libraries_payload.get("libraries", []):
        for f in lib.get("files", []):
            text = f.get("content") or f"[Binary or unsupported file: {f.get('name')}]"
            documents.append({
                "sourceType": "sharepoint_document_library",
                "sourceName": lib["name"],
                "id": f.get("id"),
                "webUrl": f.get("webUrl"),
                "text": text,
            })
    return documents


combined = {
    "source": "sharepoint_combined",
    "siteUrl": SHAREPOINT_SITE_URL,
    "exportedAt": datetime.now(timezone.utc).isoformat(),
    "documents": (
        list_items_to_documents(lists_data)
        + library_files_to_documents(libraries_data)
    ),
}

combined_path = JSON_OUTPUT_DIR / "sharepoint_combined.json"
combined_path.write_text(json.dumps(combined, indent=2, default=str), encoding="utf-8")

print(f"Combined documents: {len(combined['documents'])}")
print(f"Saved: {combined_path}")

rag_df = pd.DataFrame(combined["documents"])
rag_df.head()

Combined documents: 1223
Saved: c:\Users\natna\Documents\RAG_Project\RAG_Application-\data\sharepoint_json\sharepoint_combined.json


,sourceType,sourceName,id,webUrl,text
0,sharepoint_list,After_Action_Reports,1,https://amenhanna.sharepoint.com/Lists/After_A...,"{""@odata.etag"": ""\""95d12639-bfea-4fe3-8c7d-499..."
1,sharepoint_list,After_Action_Reports,2,https://amenhanna.sharepoint.com/Lists/After_A...,"{""@odata.etag"": ""\""585cdc8d-c6e9-4d2a-9216-28d..."
2,sharepoint_list,After_Action_Reports,3,https://amenhanna.sharepoint.com/Lists/After_A...,"{""@odata.etag"": ""\""163d8e83-3a1d-4e90-8dc0-fe8..."
3,sharepoint_list,After_Action_Reports,4,https://amenhanna.sharepoint.com/Lists/After_A...,"{""@odata.etag"": ""\""5cc0fc61-8052-4b17-a312-b14..."
4,sharepoint_list,After_Action_Reports,5,https://amenhanna.sharepoint.com/Lists/After_A...,"{""@odata.etag"": ""\""865f4974-9d6a-4ae7-8cca-53c..."


## 7. Record-aware chunking (SharePoint lists first)

**Method:** one list item = one chunk (entity/record-aware).  
Oversized items are split only as a fallback; metadata (`list`, `itemId`, `webUrl`) is kept on every part.

Document libraries (PDF / Word / PPT) will be chunked in a later step.

In [14]:
# Record-aware list chunking: 1 SharePoint list item -> 1 chunk
MAX_LIST_CHUNK_CHARS = 4000
LIST_CHUNK_OVERLAP = 200

# Noise / system fields from Graph that should not enter RAG text
SKIP_FIELD_KEYS = {
    "id",
    "ContentType",
    "ContentTypeId",
    "Edit",
    "Type",
    "ItemChildCount",
    "FolderChildCount",
    "AppAuthorLookupId",
    "AppEditorLookupId",
    "ComplianceAssetId",
    "_UIVersionString",
    "GUID",
}


def _load_lists_payload() -> dict:
    """Use in-memory lists_data if present; otherwise reload from export JSON."""
    if "lists_data" in globals() and lists_data.get("lists"):
        return lists_data
    path = JSON_OUTPUT_DIR / "sharepoint_lists.json"
    return json.loads(path.read_text(encoding="utf-8"))


def clean_list_fields(fields: dict) -> dict:
    cleaned = {}
    for key, value in (fields or {}).items():
        if key.startswith("@") or key.startswith("_"):
            continue
        if key in SKIP_FIELD_KEYS:
            continue
        if key.endswith("LookupId"):
            continue
        if value is None or value == "":
            continue
        cleaned[key] = value
    return cleaned


def fields_to_record_text(list_name: str, item_id, fields: dict) -> str:
    """Readable record text so retrieval keeps the full item context together."""
    lines = [
        f"SharePoint List: {list_name}",
        f"Item ID: {item_id}",
    ]
    for key, value in fields.items():
        if isinstance(value, (dict, list)):
            value = json.dumps(value, default=str)
        lines.append(f"{key}: {value}")
    return "\n".join(lines)


def split_oversized_text(text: str, max_chars: int, overlap: int) -> list[str]:
    """Fallback only - used when a single list item exceeds max size."""
    if len(text) <= max_chars:
        return [text]
    parts = []
    start = 0
    step = max(max_chars - overlap, 1)
    while start < len(text):
        parts.append(text[start : start + max_chars])
        start += step
    return parts


def chunk_sharepoint_lists_record_aware(
    lists_payload: dict,
    max_chars: int = MAX_LIST_CHUNK_CHARS,
    overlap: int = LIST_CHUNK_OVERLAP,
) -> list[dict]:
    chunks: list[dict] = []

    for lst in lists_payload.get("lists", []):
        list_name = lst.get("displayName", "Unknown List")
        for item in lst.get("items", []):
            fields = clean_list_fields(item.get("fields", {}))
            if not fields:
                continue

            record_text = fields_to_record_text(list_name, item.get("id"), fields)
            parts = split_oversized_text(record_text, max_chars, overlap)

            for i, part in enumerate(parts):
                chunks.append({
                    "chunkingMethod": "record_aware",
                    "sourceType": "sharepoint_list",
                    "sourceName": list_name,
                    "itemId": item.get("id"),
                    "webUrl": item.get("webUrl"),
                    "chunk_id": i,
                    "chunk_count": len(parts),
                    "char_count": len(part),
                    "text": part,
                })

    return chunks


lists_payload = _load_lists_payload()
list_chunks = chunk_sharepoint_lists_record_aware(lists_payload)

list_chunks_path = JSON_OUTPUT_DIR / "sharepoint_list_chunks.json"
list_chunks_export = {
    "source": "sharepoint_list_chunks",
    "chunkingMethod": "record_aware",
    "siteUrl": SHAREPOINT_SITE_URL,
    "exportedAt": datetime.now(timezone.utc).isoformat(),
    "maxListChunkChars": MAX_LIST_CHUNK_CHARS,
    "chunkCount": len(list_chunks),
    "chunks": list_chunks,
}
list_chunks_path.write_text(
    json.dumps(list_chunks_export, indent=2, default=str),
    encoding="utf-8",
)

list_chunks_df = pd.DataFrame(list_chunks)
split_count = int((list_chunks_df["chunk_count"] > 1).sum()) if len(list_chunks_df) else 0

print(f"List chunks: {len(list_chunks)}")
print(f"Unique list items: {list_chunks_df.groupby(['sourceName', 'itemId']).ngroups if len(list_chunks_df) else 0}")
print(f"Items split (oversized fallback): {split_count}")
print(f"Saved: {list_chunks_path}")
print("\nChunks per list:")
print(list_chunks_df.groupby("sourceName").size().sort_values(ascending=False))

list_chunks_df.head()

List chunks: 1206
Unique list items: 1206
Items split (oversized fallback): 0
Saved: c:\Users\natna\Documents\RAG_Project\RAG_Application-\data\sharepoint_json\sharepoint_list_chunks.json

Chunks per list:
sourceName
After_Action_Reports    637
Registration_Data       256
Survey_Data             256
Course_Curriculum        40
RAF_DOC                  15
Migration files           2
dtype: int64


,chunkingMethod,sourceType,sourceName,itemId,webUrl,chunk_id,chunk_count,char_count,text
0,record_aware,sharepoint_list,After_Action_Reports,1,https://amenhanna.sharepoint.com/Lists/After_A...,0,1,1728,SharePoint List: After_Action_Reports\nItem ID...
1,record_aware,sharepoint_list,After_Action_Reports,2,https://amenhanna.sharepoint.com/Lists/After_A...,0,1,1766,SharePoint List: After_Action_Reports\nItem ID...
2,record_aware,sharepoint_list,After_Action_Reports,3,https://amenhanna.sharepoint.com/Lists/After_A...,0,1,1733,SharePoint List: After_Action_Reports\nItem ID...
3,record_aware,sharepoint_list,After_Action_Reports,4,https://amenhanna.sharepoint.com/Lists/After_A...,0,1,1751,SharePoint List: After_Action_Reports\nItem ID...
4,record_aware,sharepoint_list,After_Action_Reports,5,https://amenhanna.sharepoint.com/Lists/After_A...,0,1,1733,SharePoint List: After_Action_Reports\nItem ID...


## 8. Structure-aware chunking (document libraries)

**Method by file type:**
- **Word (.docx):** heading/section chunks, then recursive size split if needed
- **PDF (.pdf):** page chunks, then recursive size split if needed
- **PowerPoint (.pptx):** one slide = one chunk
- **Excel (.xlsx):** one sheet = one chunk (tabular text)
- **Plain text:** recursive split

Downloads each file from Graph, extracts text, then chunks. Output: `sharepoint_library_chunks.json`.


In [15]:
# Structure-aware chunking for SharePoint document libraries
from io import BytesIO

from docx import Document as DocxDocument
from openpyxl import load_workbook
from pptx import Presentation
from pypdf import PdfReader

MAX_DOC_CHUNK_CHARS = 4000
DOC_CHUNK_OVERLAP = 200
RECURSIVE_SEPARATORS = ["\n\n", "\n", ". ", " ", ""]


def _load_libraries_payload() -> dict:
    if "libraries_data" in globals() and libraries_data.get("libraries"):
        return libraries_data
    path = JSON_OUTPUT_DIR / "sharepoint_libraries.json"
    return json.loads(path.read_text(encoding="utf-8"))


def download_drive_file_bytes(drive_id: str, item_id: str) -> bytes:
    url = f"{GRAPH_BASE}/drives/{drive_id}/items/{item_id}/content"
    response = requests.get(url, headers=HEADERS, timeout=180)
    response.raise_for_status()
    return response.content


def recursive_split_text(
    text: str,
    max_chars: int = MAX_DOC_CHUNK_CHARS,
    overlap: int = DOC_CHUNK_OVERLAP,
    separators: list[str] | None = None,
) -> list[str]:
    """Split on natural boundaries first, then enforce max size."""
    text = (text or "").strip()
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]

    separators = separators if separators is not None else RECURSIVE_SEPARATORS
    sep = separators[0]
    rest = separators[1:]

    if sep == "":
        parts = []
        start = 0
        step = max(max_chars - overlap, 1)
        while start < len(text):
            parts.append(text[start : start + max_chars])
            start += step
        return parts

    pieces = text.split(sep) if sep else list(text)
    chunks: list[str] = []
    current = ""

    for piece in pieces:
        candidate = piece if not current else current + sep + piece
        if len(candidate) <= max_chars:
            current = candidate
            continue
        if current:
            chunks.append(current)
        if len(piece) > max_chars:
            chunks.extend(recursive_split_text(piece, max_chars, overlap, rest))
            current = ""
        else:
            current = piece

    if current:
        chunks.append(current)
    return chunks


def extract_docx_sections(file_bytes: bytes) -> list[dict]:
    doc = DocxDocument(BytesIO(file_bytes))
    sections: list[dict] = []
    current_title = "Introduction"
    current_parts: list[str] = []

    def flush():
        body = "\n".join(p for p in current_parts if p.strip()).strip()
        if body:
            sections.append({"unit": current_title, "text": body})

    for para in doc.paragraphs:
        style = (para.style.name if para.style else "") or ""
        text = (para.text or "").strip()
        if not text:
            continue
        if style.startswith("Heading"):
            flush()
            current_title = text
            current_parts = []
        else:
            current_parts.append(text)

    flush()
    if not sections:
        full = "\n".join(p.text for p in doc.paragraphs if (p.text or "").strip())
        if full.strip():
            sections = [{"unit": "Document", "text": full.strip()}]
    return sections


def extract_pdf_pages(file_bytes: bytes) -> list[dict]:
    reader = PdfReader(BytesIO(file_bytes))
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        if text:
            pages.append({"unit": f"Page {i}", "text": text})
    return pages


def extract_pptx_slides(file_bytes: bytes) -> list[dict]:
    prs = Presentation(BytesIO(file_bytes))
    slides = []
    for i, slide in enumerate(prs.slides, start=1):
        texts = []
        title = None
        if slide.shapes.title and slide.shapes.title.text:
            title = slide.shapes.title.text.strip()
        for shape in slide.shapes:
            if not hasattr(shape, "text"):
                continue
            value = (shape.text or "").strip()
            if not value:
                continue
            if title and value == title:
                continue
            texts.append(value)
        body = "\n".join(texts).strip()
        unit = title or f"Slide {i}"
        combined = f"{title}\n{body}".strip() if title else body
        if combined:
            slides.append({"unit": unit, "text": combined})
    return slides


def extract_xlsx_sheets(file_bytes: bytes) -> list[dict]:
    wb = load_workbook(BytesIO(file_bytes), read_only=True, data_only=True)
    sheets = []
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        rows = []
        for row in ws.iter_rows(values_only=True):
            values = ["" if v is None else str(v) for v in row]
            if any(v.strip() for v in values):
                rows.append(" | ".join(values))
        text = "\n".join(rows).strip()
        if text:
            sheets.append({"unit": f"Sheet: {sheet_name}", "text": text})
    wb.close()
    return sheets


def extract_plain_text(file_bytes: bytes) -> list[dict]:
    try:
        text = file_bytes.decode("utf-8")
    except UnicodeDecodeError:
        text = file_bytes.decode("latin-1", errors="ignore")
    text = text.strip()
    return [{"unit": "Document", "text": text}] if text else []


def extract_structural_units(file_name: str, file_bytes: bytes) -> list[dict]:
    ext = Path(file_name).suffix.lower()
    if ext == ".docx":
        return extract_docx_sections(file_bytes)
    if ext == ".pdf":
        return extract_pdf_pages(file_bytes)
    if ext == ".pptx":
        return extract_pptx_slides(file_bytes)
    if ext in {".xlsx", ".xlsm"}:
        return extract_xlsx_sheets(file_bytes)
    if ext in {".txt", ".md", ".csv", ".json", ".html", ".htm"}:
        return extract_plain_text(file_bytes)
    return []


def chunk_document_libraries_structure_aware(
    libraries_payload: dict,
    max_chars: int = MAX_DOC_CHUNK_CHARS,
    overlap: int = DOC_CHUNK_OVERLAP,
) -> list[dict]:
    chunks: list[dict] = []

    for lib in libraries_payload.get("libraries", []):
        library_name = lib.get("name", "Unknown Library")
        drive_id = lib.get("driveId")
        for f in lib.get("files", []):
            file_name = f.get("name", "unknown")
            ext = Path(file_name).suffix.lower()
            item_id = f.get("id")
            print(f"Processing: {library_name} / {file_name}")

            try:
                file_bytes = download_drive_file_bytes(drive_id, item_id)
                units = extract_structural_units(file_name, file_bytes)
            except Exception as exc:
                print(f"  SKIPPED ({exc})")
                continue

            if not units:
                print("  No extractable text")
                continue

            method = {
                ".docx": "section_recursive",
                ".pdf": "page_recursive",
                ".pptx": "slide",
                ".xlsx": "sheet_recursive",
                ".xlsm": "sheet_recursive",
            }.get(ext, "recursive")

            file_chunk_index = 0
            for unit in units:
                if ext == ".pptx" and len(unit["text"]) <= max_chars:
                    parts = [unit["text"]]
                else:
                    parts = recursive_split_text(unit["text"], max_chars, overlap)

                for part_i, part in enumerate(parts):
                    header = (
                        f"Document Library: {library_name}\n"
                        f"File: {file_name}\n"
                        f"Section: {unit['unit']}\n\n"
                    )
                    text = header + part
                    chunks.append({
                        "chunkingMethod": method,
                        "sourceType": "sharepoint_document_library",
                        "sourceName": library_name,
                        "fileName": file_name,
                        "fileId": item_id,
                        "webUrl": f.get("webUrl"),
                        "unit": unit["unit"],
                        "unit_part": part_i,
                        "chunk_id": file_chunk_index,
                        "char_count": len(text),
                        "text": text,
                    })
                    file_chunk_index += 1

            print(f"  -> {file_chunk_index} chunk(s)")

    return chunks


libraries_payload = _load_libraries_payload()
library_chunks = chunk_document_libraries_structure_aware(libraries_payload)

library_chunks_path = JSON_OUTPUT_DIR / "sharepoint_library_chunks.json"
library_chunks_export = {
    "source": "sharepoint_library_chunks",
    "chunkingMethod": "structure_aware",
    "siteUrl": SHAREPOINT_SITE_URL,
    "exportedAt": datetime.now(timezone.utc).isoformat(),
    "maxDocChunkChars": MAX_DOC_CHUNK_CHARS,
    "chunkCount": len(library_chunks),
    "chunks": library_chunks,
}
library_chunks_path.write_text(
    json.dumps(library_chunks_export, indent=2, default=str),
    encoding="utf-8",
)

library_chunks_df = pd.DataFrame(library_chunks)
print(f"\nLibrary chunks: {len(library_chunks)}")
print(f"Unique files: {library_chunks_df['fileName'].nunique() if len(library_chunks_df) else 0}")
print(f"Saved: {library_chunks_path}")
if len(library_chunks_df):
    print("\nChunks by method:")
    print(library_chunks_df.groupby("chunkingMethod").size().sort_values(ascending=False))
    print("\nChunks per library:")
    print(library_chunks_df.groupby("sourceName").size().sort_values(ascending=False))

library_chunks_df.head()


Processing: RAF_DOC / CTC_Comprehensive_Report.docx
  -> 7 chunk(s)
Processing: RAF_DOC / CTC_Comprehensive_Report.pdf
  -> 6 chunk(s)
Processing: RAF_DOC / CTC_Presentation.pptx
  -> 4 chunk(s)
Processing: RAF_DOC / CWO_Comprehensive_Report.docx
  -> 7 chunk(s)
Processing: RAF_DOC / CWO_Comprehensive_Report.pdf
  -> 6 chunk(s)
Processing: RAF_DOC / CWO_Presentation.pptx
  -> 4 chunk(s)
Processing: RAF_DOC / PLC_Comprehensive_Report.docx
  -> 7 chunk(s)
Processing: RAF_DOC / PLC_Comprehensive_Report.pdf
  -> 6 chunk(s)
Processing: RAF_DOC / PLC_Presentation.pptx
  -> 4 chunk(s)
Processing: RAF_DOC / SLC_Comprehensive_Report.docx
  -> 7 chunk(s)
Processing: RAF_DOC / SLC_Comprehensive_Report.pdf
  -> 6 chunk(s)
Processing: RAF_DOC / SLC_Presentation.pptx
  -> 4 chunk(s)
Processing: RAF_DOC / TLC_Comprehensive_Report.docx
  -> 7 chunk(s)
Processing: RAF_DOC / TLC_Comprehensive_Report.pdf
  -> 6 chunk(s)
Processing: RAF_DOC / TLC_Presentation.pptx
  -> 4 chunk(s)
Processing: Migration fil

,chunkingMethod,sourceType,sourceName,fileName,fileId,webUrl,unit,unit_part,chunk_id,char_count,text
0,section_recursive,sharepoint_document_library,RAF_DOC,CTC_Comprehensive_Report.docx,01EGXJNRZYL2NWNPFH7ND37XKX7NALWJIU,https://amenhanna.sharepoint.com/_layouts/15/D...,Introduction,0,0,211,Document Library: RAF_DOC\nFile: CTC_Comprehen...
1,section_recursive,sharepoint_document_library,RAF_DOC,CTC_Comprehensive_Report.docx,01EGXJNRZYL2NWNPFH7ND37XKX7NALWJIU,https://amenhanna.sharepoint.com/_layouts/15/D...,EXECUTIVE SUMMARY,0,1,539,Document Library: RAF_DOC\nFile: CTC_Comprehen...
2,section_recursive,sharepoint_document_library,RAF_DOC,CTC_Comprehensive_Report.docx,01EGXJNRZYL2NWNPFH7ND37XKX7NALWJIU,https://amenhanna.sharepoint.com/_layouts/15/D...,COURSE OVERVIEW,0,2,156,Document Library: RAF_DOC\nFile: CTC_Comprehen...
3,section_recursive,sharepoint_document_library,RAF_DOC,CTC_Comprehensive_Report.docx,01EGXJNRZYL2NWNPFH7ND37XKX7NALWJIU,https://amenhanna.sharepoint.com/_layouts/15/D...,CURRICULUM DETAILS,0,3,208,Document Library: RAF_DOC\nFile: CTC_Comprehen...
4,section_recursive,sharepoint_document_library,RAF_DOC,CTC_Comprehensive_Report.docx,01EGXJNRZYL2NWNPFH7ND37XKX7NALWJIU,https://amenhanna.sharepoint.com/_layouts/15/D...,Strengths to Maintain,0,4,458,Document Library: RAF_DOC\nFile: CTC_Comprehen...


## 9. Embeddings + Chroma (free local vector RAG)

**Stack:** Sentence Transformers (`all-MiniLM-L6-v2`) + Chroma persistent store.

Loads list + library chunks, embeds them locally, and stores vectors under `vectorstore/chroma`.  
Video/audio can be added later as timed transcript chunks with the same schema (`modality`, timestamps).


In [19]:
# Sentence Transformers + Chroma: index list and library chunks
import chromadb
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
CHROMA_DIR = PROJECT_DIR / "vectorstore" / "chroma"
COLLECTION_NAME = "nlead_chunks"
EMBED_BATCH_SIZE = 64
RESET_COLLECTION = True  # set False to append without wiping


def _load_chunk_records() -> list[dict]:
    records: list[dict] = []

    list_path = JSON_OUTPUT_DIR / "sharepoint_list_chunks.json"
    if list_path.exists():
        payload = json.loads(list_path.read_text(encoding="utf-8"))
        for c in payload.get("chunks", []):
            records.append({
                "text": c.get("text", ""),
                "modality": "list",
                "sourceType": c.get("sourceType", "sharepoint_list"),
                "sourceName": c.get("sourceName", ""),
                "webUrl": c.get("webUrl") or "",
                "itemId": str(c.get("itemId") or ""),
                "fileName": "",
                "fileId": "",
                "unit": "",
                "chunkingMethod": c.get("chunkingMethod", "record_aware"),
                "chunk_id": int(c.get("chunk_id", 0)),
                "timestamp_start": "",
                "timestamp_end": "",
            })

    lib_path = JSON_OUTPUT_DIR / "sharepoint_library_chunks.json"
    if lib_path.exists():
        payload = json.loads(lib_path.read_text(encoding="utf-8"))
        for c in payload.get("chunks", []):
            records.append({
                "text": c.get("text", ""),
                "modality": "document",
                "sourceType": c.get("sourceType", "sharepoint_document_library"),
                "sourceName": c.get("sourceName", ""),
                "webUrl": c.get("webUrl") or "",
                "itemId": "",
                "fileName": c.get("fileName") or "",
                "fileId": str(c.get("fileId") or ""),
                "unit": c.get("unit") or "",
                "chunkingMethod": c.get("chunkingMethod", "structure_aware"),
                "chunk_id": int(c.get("chunk_id", 0)),
                "timestamp_start": "",
                "timestamp_end": "",
            })

    # Drop empty text
    return [r for r in records if (r.get("text") or "").strip()]


def _chroma_metadata(record: dict) -> dict:
    # Chroma metadata must be str/int/float/bool
    return {
        "modality": record["modality"],
        "sourceType": record["sourceType"],
        "sourceName": record["sourceName"],
        "webUrl": record["webUrl"],
        "itemId": record["itemId"],
        "fileName": record["fileName"],
        "fileId": record["fileId"],
        "unit": record["unit"],
        "chunkingMethod": record["chunkingMethod"],
        "chunk_id": record["chunk_id"],
        "timestamp_start": record["timestamp_start"],
        "timestamp_end": record["timestamp_end"],
    }


chunk_records = _load_chunk_records()
print(f"Chunks to index: {len(chunk_records)}")
print(f"  lists: {sum(1 for r in chunk_records if r['modality'] == 'list')}")
print(f"  documents: {sum(1 for r in chunk_records if r['modality'] == 'document')}")

print(f"Loading embedding model: {EMBED_MODEL_NAME}")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

CHROMA_DIR.mkdir(parents=True, exist_ok=True)
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

if RESET_COLLECTION:
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
        print(f"Deleted existing collection: {COLLECTION_NAME}")
    except Exception:
        pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

ids: list[str] = []
documents: list[str] = []
metadatas: list[dict] = []

for i, record in enumerate(chunk_records):
    ids.append(f"{record['modality']}_{record['sourceName']}_{record['itemId'] or record['fileId']}_{record['chunk_id']}_{i}")
    documents.append(record["text"])
    metadatas.append(_chroma_metadata(record))

print("Embedding + upserting into Chroma...")
for start in range(0, len(documents), EMBED_BATCH_SIZE):
    end = min(start + EMBED_BATCH_SIZE, len(documents))
    batch_docs = documents[start:end]
    batch_ids = ids[start:end]
    batch_meta = metadatas[start:end]
    batch_embeddings = embed_model.encode(
        batch_docs,
        show_progress_bar=False,
        normalize_embeddings=True,
    ).tolist()
    collection.upsert(
        ids=batch_ids,
        documents=batch_docs,
        metadatas=batch_meta,
        embeddings=batch_embeddings,
    )
    print(f"  indexed {end}/{len(documents)}")

print(f"Done. Collection '{COLLECTION_NAME}' count: {collection.count()}")
print(f"Chroma path: {CHROMA_DIR}")


Chunks to index: 3390
  lists: 1206
  documents: 2184
Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Deleted existing collection: nlead_chunks
Embedding + upserting into Chroma...
  indexed 64/3390
  indexed 128/3390
  indexed 192/3390
  indexed 256/3390
  indexed 320/3390
  indexed 384/3390
  indexed 448/3390
  indexed 512/3390
  indexed 576/3390
  indexed 640/3390
  indexed 704/3390
  indexed 768/3390
  indexed 832/3390
  indexed 896/3390
  indexed 960/3390
  indexed 1024/3390
  indexed 1088/3390
  indexed 1152/3390
  indexed 1216/3390
  indexed 1280/3390
  indexed 1344/3390
  indexed 1408/3390
  indexed 1472/3390
  indexed 1536/3390
  indexed 1600/3390
  indexed 1664/3390
  indexed 1728/3390
  indexed 1792/3390
  indexed 1856/3390
  indexed 1920/3390
  indexed 1984/3390
  indexed 2048/3390
  indexed 2112/3390
  indexed 2176/3390
  indexed 2240/3390
  indexed 2304/3390
  indexed 2368/3390
  indexed 2432/3390
  indexed 2496/3390
  indexed 2560/3390
  indexed 2624/3390
  indexed 2688/3390
  indexed 2752/3390
  indexed 2816/3390
  indexed 2880/3390
  indexed 2944/3390
  indexed 3008/33

In [28]:
# Test retrieval (no LLM yet — just top matching chunks)
def retrieve_chunks(
    query: str,
    n_results: int = 5,
    modality: str | None = None,
) -> pd.DataFrame:
    """Semantic search over the Chroma collection."""
    query_embedding = embed_model.encode(
        [query],
        normalize_embeddings=True,
    ).tolist()

    where = {"modality": modality} if modality else None
    result = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        where=where,
        include=["documents", "metadatas", "distances"],
    )

    rows = []
    for doc, meta, dist in zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0],
    ):
        rows.append({
            "distance": round(float(dist), 4),
            "modality": meta.get("modality"),
            "sourceName": meta.get("sourceName"),
            "fileName": meta.get("fileName"),
            "unit": meta.get("unit"),
            "webUrl": meta.get("webUrl"),
            "text_preview": (doc or "")[:240].replace("\n", " "),
        })
    return pd.DataFrame(rows)


# Example queries — change these to explore retrieval quality
query = "leadership fundamentals and conflict resolution feedback"
hits = retrieve_chunks(query, n_results=5)
print(f"Query: {query}\n")
hits


Query: leadership fundamentals and conflict resolution feedback



,distance,modality,sourceName,fileName,unit,webUrl,text_preview
0,0.5780,list,RAF_DOC,,,https://amenhanna.sharepoint.com/RAF_DOC/SLC_C...,SharePoint List: RAF_DOC Item ID: 13 FileLeafR...
1,0.5869,list,After_Action_Reports,,,https://amenhanna.sharepoint.com/Lists/After_A...,SharePoint List: After_Action_Reports Item ID:...
2,0.5921,list,After_Action_Reports,,,https://amenhanna.sharepoint.com/Lists/After_A...,SharePoint List: After_Action_Reports Item ID:...
3,0.5945,list,RAF_DOC,,,https://amenhanna.sharepoint.com/RAF_DOC/TLC_C...,SharePoint List: RAF_DOC Item ID: 15 FileLeafR...
4,0.5947,list,After_Action_Reports,,,https://amenhanna.sharepoint.com/Lists/After_A...,SharePoint List: After_Action_Reports Item ID:...


## 10. Ask a question (retrieve + LLM answer)

Uses Chroma retrieval, then an LLM to answer with citations.

**Current setup: Ollama (local, free, no quota)**
1. Install from https://ollama.com
2. In a terminal: `ollama pull llama3.2`
3. Keep Ollama running, then run the cells below

`.env` should have `LLM_PROVIDER=ollama` (Gemini remains an optional backup).


In [30]:
# Retrieve + LLM answer helpers (Ollama or Gemini)
load_dotenv(override=True)

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "auto").strip().lower()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY") or ""
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.0-flash-lite")
MAX_SOURCE_CHARS = int(os.getenv("RAG_MAX_SOURCE_CHARS", "1200"))
GEMINI_RETRY_SECONDS = int(os.getenv("GEMINI_RETRY_SECONDS", "35"))


def retrieve_context(
    query: str,
    n_results: int = 4,
    modality: str | None = None,
) -> list[dict]:
    """Return top chunks with full text + metadata for the LLM prompt."""
    query_embedding = embed_model.encode([query], normalize_embeddings=True).tolist()
    where = {"modality": modality} if modality else None
    result = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        where=where,
        include=["documents", "metadatas", "distances"],
    )

    contexts = []
    for i, (doc, meta, dist) in enumerate(
        zip(result["documents"][0], result["metadatas"][0], result["distances"][0]),
        start=1,
    ):
        contexts.append({
            "ref": i,
            "distance": float(dist),
            "text": doc or "",
            "modality": meta.get("modality", ""),
            "sourceName": meta.get("sourceName", ""),
            "fileName": meta.get("fileName", ""),
            "unit": meta.get("unit", ""),
            "webUrl": meta.get("webUrl", ""),
            "itemId": meta.get("itemId", ""),
        })
    return contexts


def build_rag_prompt(question: str, contexts: list[dict]) -> str:
    blocks = []
    for c in contexts:
        label = c["sourceName"]
        if c.get("fileName"):
            label = f"{label} / {c['fileName']}"
        if c.get("unit"):
            label = f"{label} [{c['unit']}]"
        body = c["text"] or ""
        if len(body) > MAX_SOURCE_CHARS:
            body = body[:MAX_SOURCE_CHARS] + "\n...[truncated]..."
        blocks.append(
            f"[Source {c['ref']}] ({c['modality']}) {label}\n"
            f"URL: {c.get('webUrl') or 'n/a'}\n"
            f"{body}"
        )

    context_text = "\n\n---\n\n".join(blocks)
    return (
        "You are a helpful assistant for the NLEAD SharePoint knowledge base.\n"
        "Answer the question using ONLY the sources below.\n"
        "If the sources are not enough, say you do not have enough information.\n"
        "Cite sources inline like [Source 1], [Source 2].\n"
        "Keep the answer clear and concise.\n\n"
        f"SOURCES:\n{context_text}\n\n"
        f"QUESTION: {question}\n\n"
        "ANSWER:"
    )


def _ollama_available() -> bool:
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=3)
        return r.status_code == 200
    except Exception:
        return False


def _call_ollama(prompt: str) -> str:
    url = f"{OLLAMA_BASE_URL}/api/chat"
    payload = {
        "model": OLLAMA_MODEL,
        "messages": [
            {"role": "system", "content": "Answer only from provided sources and cite them."},
            {"role": "user", "content": prompt},
        ],
        "stream": False,
    }
    r = requests.post(url, json=payload, timeout=180)
    r.raise_for_status()
    data = r.json()
    return data.get("message", {}).get("content", "").strip()


def _call_gemini(prompt: str, retries: int = 1) -> str:
    import time

    if not GEMINI_API_KEY:
        raise RuntimeError("GEMINI_API_KEY / GOOGLE_API_KEY is not set in .env")
    url = (
        f"https://generativelanguage.googleapis.com/v1beta/models/"
        f"{GEMINI_MODEL}:generateContent"
    )
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": 0.2},
    }

    last_error = ""
    for attempt in range(retries + 1):
        r = requests.post(url, params={"key": GEMINI_API_KEY}, json=payload, timeout=180)
        if r.status_code == 200:
            data = r.json()
            candidates = data.get("candidates") or []
            if not candidates:
                raise RuntimeError(f"Gemini returned no candidates: {data}")
            parts = candidates[0].get("content", {}).get("parts") or []
            return "".join(p.get("text", "") for p in parts).strip()

        last_error = r.text[:400]
        if r.status_code == 429 and attempt < retries:
            print(f"Gemini quota/rate limit hit. Waiting {GEMINI_RETRY_SECONDS}s then retrying...")
            time.sleep(GEMINI_RETRY_SECONDS)
            continue

        if r.status_code == 429:
            raise RuntimeError(
                "Gemini free-tier quota exceeded (HTTP 429).\n"
                "Fixes:\n"
                "  1. Wait 1 minute (RPM) or until tomorrow (daily quota), then retry\n"
                "  2. In .env set GEMINI_MODEL=gemini-2.0-flash-lite\n"
                "  3. Ask with fewer sources: ask_rag(question, n_results=3)\n"
                "  4. Or switch to local Ollama (free, no quota)\n"
                f"Details: {last_error}"
            )
        raise RuntimeError(f"Gemini error {r.status_code}: {last_error}")

    raise RuntimeError(f"Gemini failed: {last_error}")


def resolve_llm_provider() -> str:
    if LLM_PROVIDER in {"ollama", "gemini"}:
        return LLM_PROVIDER
    if _ollama_available():
        return "ollama"
    if GEMINI_API_KEY:
        return "gemini"
    raise RuntimeError(
        "No LLM available.\n"
        "Option A (local/free): install Ollama from https://ollama.com then run:\n"
        f"  ollama pull {OLLAMA_MODEL}\n"
        "Option B (cloud free tier): create a Gemini API key and set in .env:\n"
        "  GEMINI_API_KEY=your_key\n"
        "  LLM_PROVIDER=gemini"
    )


def ask_rag(
    question: str,
    n_results: int = 4,
    modality: str | None = None,
) -> dict:
    """Full RAG: retrieve -> prompt LLM -> return answer + sources."""
    provider = resolve_llm_provider()
    contexts = retrieve_context(question, n_results=n_results, modality=modality)
    prompt = build_rag_prompt(question, contexts)

    if provider == "ollama":
        answer = _call_ollama(prompt)
        model_name = OLLAMA_MODEL
    else:
        answer = _call_gemini(prompt)
        model_name = GEMINI_MODEL

    return {
        "question": question,
        "provider": provider,
        "model": model_name,
        "answer": answer,
        "sources": contexts,
    }


provider = resolve_llm_provider()
print(f"LLM ready: {provider}")
if provider == "ollama":
    print(f"  model: {OLLAMA_MODEL} @ {OLLAMA_BASE_URL}")
else:
    print(f"  model: {GEMINI_MODEL}")


LLM ready: ollama
  model: llama3.2 @ http://localhost:11434


In [31]:
# Ask any question (not predefined)
question = "What feedback themes appear around leadership and conflict resolution?"

result = ask_rag(question, n_results=3)

print(f"Provider: {result['provider']} | Model: {result['model']}")
print(f"\nQ: {result['question']}\n")
print("A:", result["answer"])
print("\nSources used:")
for s in result["sources"]:
    label = s["sourceName"]
    if s.get("fileName"):
        label += f" / {s['fileName']}"
    if s.get("unit"):
        label += f" [{s['unit']}]"
    print(f"  [{s['ref']}] {s['modality']} | {label} | distance={s['distance']:.4f}")
    if s.get("webUrl"):
        print(f"       {s['webUrl']}")


Provider: ollama | Model: llama3.2

Q: What feedback themes appear around leadership and conflict resolution?

A: [Source 1] [Source 2]

The available sources provide some insight into the content of the courses, but they do not explicitly mention specific feedback themes related to leadership and conflict resolution.

However, I can infer that [Source 1] provides general information about the Senior Leadership Course (SLC) and Top Leadership Course (TLC), which may include aspects of leadership. 

For example, it mentions that the Command Team Course (CTC) "emphasizes tactical decision-making, unit cohesion, command climate development, and fleet integration at the command level" [Source 1]. This suggests that the courses may cover topics related to effective command leadership.

Regarding conflict resolution, I do not have enough information from the provided sources.

Sources used:
  [1] list | After_Action_Reports | distance=0.5900
       https://amenhanna.sharepoint.com/Lists/Afte